# 임베딩 거리 기반 이상점수 비교

저장된 분류 앙상블의 128차원 임베딩을 이용해 기계별 정상 분포와의 Mahalanobis 거리를 계산한다. 기존 softmax 이상점수와 같은 테스트 샘플에서 비교하며, 차이의 불확실성은 paired bootstrap 95% 신뢰구간으로 확인한다.

- 모델: `models/idclf_0.pth` ~ `idclf_4.pth` 재사용(재학습 없음)
- 정상 분포: split seed 42의 학습 정상만 사용
- 공분산: 대각 공분산 + 사전 고정 shrinkage 0.1
- 앙상블: 모델별 샘플 점수를 먼저 평균한 뒤 AUC 계산
- 결합 점수: 검증 기반 가중치가 없으므로 테스트 선택편향 방지를 위해 제외


In [1]:
## [0] 준비
# 노트북 위치와 관계없이 src 모듈을 불러올 수 있도록 프로젝트 루트를 추가한다.
import sys
sys.path.insert(0, '..')

# Windows에서 torch와 Matplotlib의 OpenMP 라이브러리 충돌을 막는다.
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import numpy as np
import torch
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.metrics import roc_auc_score

from src import data, evaluate
from src.model import IDClassifier

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SPLIT_SEED = 42
MODEL_SEEDS = [0, 1, 2, 3, 4]
MACHINES = ['id_00', 'id_02', 'id_04', 'id_06']
SHRINKAGE_ALPHA = 0.1
NUMERICAL_EPSILON = 1e-12
BOOTSTRAP_ITERATIONS = 2000
BOOTSTRAP_SEED = 42
MODEL_DIR = '../models'
ASSET_PATH = '../assets/embedding_score_comparison.png'

print('평가 장치 :', DEVICE)
print('고정 shrinkage :', SHRINKAGE_ALPHA)


평가 장치 : cuda
고정 shrinkage : 0.1


In [2]:
## [1] 거리와 paired bootstrap 헬퍼
def diagonal_mahalanobis(trainEmbeddingNP, targetEmbeddingNP):
    """학습 정상의 대각 공분산으로 target의 제곱 Mahalanobis 거리를 계산한다."""
    meanNP = trainEmbeddingNP.mean(axis=0)
    varianceNP = trainEmbeddingNP.var(axis=0, ddof=1)

    # 각 차원의 분산을 전체 평균 분산 쪽으로 당겨 작은 분산의 폭주를 막는다.
    mean_variance = varianceNP.mean()
    regularizedVarianceNP = (
        (1 - SHRINKAGE_ALPHA) * varianceNP
        + SHRINKAGE_ALPHA * mean_variance
        + NUMERICAL_EPSILON
    )
    differenceNP = targetEmbeddingNP - meanNP
    return np.sum((differenceNP ** 2) / regularizedVarianceNP, axis=1)


def paired_bootstrap_auc_difference(normalSoftNP, abnormalSoftNP,
                                    normalEmbeddingNP, abnormalEmbeddingNP):
    """동일 샘플 재추출로 임베딩 AUC-softmax AUC 차이의 95% CI를 계산한다."""
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    difference_list = []
    normal_count = len(normalSoftNP)
    abnormal_count = len(abnormalSoftNP)
    yTrueNP = np.array([0] * normal_count + [1] * abnormal_count)

    for iteration in range(BOOTSTRAP_ITERATIONS):
        normalIndexNP = rng.integers(0, normal_count, normal_count)
        abnormalIndexNP = rng.integers(0, abnormal_count, abnormal_count)
        softScoreNP = np.concatenate([
            normalSoftNP[normalIndexNP], abnormalSoftNP[abnormalIndexNP]
        ])
        embeddingScoreNP = np.concatenate([
            normalEmbeddingNP[normalIndexNP], abnormalEmbeddingNP[abnormalIndexNP]
        ])
        soft_auc = roc_auc_score(yTrueNP, softScoreNP)
        embedding_auc = roc_auc_score(yTrueNP, embeddingScoreNP)
        difference_list.append(embedding_auc - soft_auc)

    differenceNP = np.array(difference_list)
    lower = np.percentile(differenceNP, 2.5)
    upper = np.percentile(differenceNP, 97.5)
    return lower, upper


In [3]:
## [2] 고정 학습/테스트 데이터와 저장 모델 로드
# 학습 정상만 분포 추정에 쓰며, 테스트 정상·이상은 마지막 평가에만 사용한다.
X_trainNP, y_trainNP, tests = data.load_for_classification(seed=SPLIT_SEED)
model_list = []

for model_seed in MODEL_SEEDS:
    model = IDClassifier().to(DEVICE)
    model_path = os.path.join(MODEL_DIR, f'idclf_{model_seed}.pth')
    state_dict = torch.load(model_path, map_location=DEVICE, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()
    model_list.append(model)

print('학습 정상 shape :', X_trainNP.shape)
for machine in MACHINES:
    machine_idx, testNormalNP, testAbnormalNP = tests[machine]
    train_count = int(np.sum(y_trainNP == machine_idx))
    print(machine, 'train/정상 test/이상 test :',
          train_count, len(testNormalNP), len(testAbnormalNP))
print('저장 모델 수 :', len(model_list))


학습 정상 shape : (2562, 64, 313)
id_00 train/정상 test/이상 test : 854 214 356
id_02 train/정상 test/이상 test : 854 214 267
id_04 train/정상 test/이상 test : 427 107 178
id_06 train/정상 test/이상 test : 427 107 89
저장 모델 수 : 5


In [4]:
## [3] 모델별 점수 계산 후 샘플 단위 앙상블 — held-out test 1회
# 이 셀을 실행하기 전에 거리 정의와 shrinkage를 모두 고정했다.
softNormalByMachine = {machine: [] for machine in MACHINES}
softAbnormalByMachine = {machine: [] for machine in MACHINES}
embeddingNormalByMachine = {machine: [] for machine in MACHINES}
embeddingAbnormalByMachine = {machine: [] for machine in MACHINES}

for model_number in range(len(model_list)):
    model = model_list[model_number]
    trainEmbeddingNP = evaluate.extract_embeddings(model, X_trainNP, DEVICE)

    for machine in MACHINES:
        machine_idx, testNormalNP, testAbnormalNP = tests[machine]
        machineTrainEmbeddingNP = trainEmbeddingNP[y_trainNP == machine_idx]

        normalProbaNP = evaluate.predict_proba(model, testNormalNP, DEVICE)
        abnormalProbaNP = evaluate.predict_proba(model, testAbnormalNP, DEVICE)
        softNormalByMachine[machine].append(1 - normalProbaNP[:, machine_idx])
        softAbnormalByMachine[machine].append(1 - abnormalProbaNP[:, machine_idx])

        normalEmbeddingNP = evaluate.extract_embeddings(model, testNormalNP, DEVICE)
        abnormalEmbeddingNP = evaluate.extract_embeddings(model, testAbnormalNP, DEVICE)
        embeddingNormalByMachine[machine].append(
            diagonal_mahalanobis(machineTrainEmbeddingNP, normalEmbeddingNP)
        )
        embeddingAbnormalByMachine[machine].append(
            diagonal_mahalanobis(machineTrainEmbeddingNP, abnormalEmbeddingNP)
        )

    print('모델', MODEL_SEEDS[model_number], '점수 계산 완료')


모델 0 점수 계산 완료
모델 1 점수 계산 완료
모델 2 점수 계산 완료
모델 3 점수 계산 완료
모델 4 점수 계산 완료


In [5]:
## [4] AUC와 paired bootstrap 95% 신뢰구간
results = {}
print('기계 | softmax AUC | 임베딩 AUC | 차이 | paired bootstrap 95% CI')
print('-' * 78)

for machine in MACHINES:
    softNormalNP = np.mean(np.array(softNormalByMachine[machine]), axis=0)
    softAbnormalNP = np.mean(np.array(softAbnormalByMachine[machine]), axis=0)
    embeddingNormalNP = np.mean(np.array(embeddingNormalByMachine[machine]), axis=0)
    embeddingAbnormalNP = np.mean(np.array(embeddingAbnormalByMachine[machine]), axis=0)

    soft_auc = evaluate.compute_auc(softNormalNP, softAbnormalNP)
    embedding_auc = evaluate.compute_auc(embeddingNormalNP, embeddingAbnormalNP)
    difference = embedding_auc - soft_auc
    lower, upper = paired_bootstrap_auc_difference(
        softNormalNP, softAbnormalNP, embeddingNormalNP, embeddingAbnormalNP
    )
    results[machine] = {
        'softmax': soft_auc,
        'embedding': embedding_auc,
        'difference': difference,
        'ci_lower': lower,
        'ci_upper': upper,
    }
    print(f'{machine} | {soft_auc:.4f} | {embedding_auc:.4f} | '          f'{difference:+.4f} | [{lower:+.4f}, {upper:+.4f}]')

print('\n판정 기준: 신뢰구간 전체가 0보다 클 때만 통계적으로 명확한 개선으로 본다.')


기계 | softmax AUC | 임베딩 AUC | 차이 | paired bootstrap 95% CI
------------------------------------------------------------------------------
id_00 | 0.9904 | 0.9935 | +0.0031 | [-0.0009, +0.0076]
id_02 | 0.9572 | 0.9582 | +0.0010 | [-0.0166, +0.0208]
id_04 | 0.9977 | 0.9984 | +0.0006 | [-0.0007, +0.0026]
id_06 | 0.8065 | 0.7527 | -0.0538 | [-0.1055, -0.0026]

판정 기준: 신뢰구간 전체가 0보다 클 때만 통계적으로 명확한 개선으로 본다.


In [6]:
## [5] 포트폴리오용 비교 시각화
xNP = np.arange(len(MACHINES))
softAucNP = np.array([results[machine]['softmax'] for machine in MACHINES])
embeddingAucNP = np.array([results[machine]['embedding'] for machine in MACHINES])
differenceNP = np.array([results[machine]['difference'] for machine in MACHINES])
lowerErrorNP = differenceNP - np.array([results[machine]['ci_lower'] for machine in MACHINES])
upperErrorNP = np.array([results[machine]['ci_upper'] for machine in MACHINES]) - differenceNP

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
width = 0.36
axes[0].bar(xNP - width / 2, softAucNP, width, label='Softmax 점수')
axes[0].bar(xNP + width / 2, embeddingAucNP, width, label='임베딩 거리')
axes[0].set_xticks(xNP)
axes[0].set_xticklabels(MACHINES)
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel('ROC AUC')
axes[0].set_title('기계별 앙상블 이상탐지 성능')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.25)

barColorLST = ['tab:blue' if value >= 0 else 'tab:red' for value in differenceNP]
axes[1].bar(xNP, differenceNP, color=barColorLST, alpha=0.8)
axes[1].errorbar(xNP, differenceNP, yerr=[lowerErrorNP, upperErrorNP],
                 fmt='none', ecolor='black', capsize=5)
axes[1].axhline(0, color='black', linewidth=1)
axes[1].set_xticks(xNP)
axes[1].set_xticklabels(MACHINES)
axes[1].set_ylabel('AUC 차이 (임베딩 - Softmax)')
axes[1].set_title('Paired bootstrap 95% 신뢰구간')
axes[1].grid(axis='y', alpha=0.25)

plt.tight_layout()
plt.savefig(ASSET_PATH, dpi=150, bbox_inches='tight')
plt.show()
print('그래프 저장 :', ASSET_PATH)


19_embedding_score.ipynb:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  "\n",
그래프 저장 : ../assets/embedding_score_comparison.png


## 결과 판정

- id_00: 0.9904 → 0.9935, 차이 +0.0031, 95% CI [-0.0009, +0.0076]
- id_02: 0.9572 → 0.9582, 차이 +0.0010, 95% CI [-0.0166, +0.0208]
- id_04: 0.9977 → 0.9984, 차이 +0.0006, 95% CI [-0.0007, +0.0026]
- id_06: 0.8065 → 0.7527, 차이 -0.0538, 95% CI [-0.1055, -0.0026]

id_00/02/04의 작은 상승은 신뢰구간이 0을 포함하므로 확정 개선이 아니다. 핵심 id_06은 신뢰구간 전체가 0보다 작아 임베딩 거리가 softmax보다 유의하게 나빴다. 따라서 이번 고정 방법은 채택하지 않고 기존 softmax 앙상블을 유지한다.


## 해석 시 제한사항

- 이상 샘플 수가 적다는 사실은 낮은 성능의 직접 원인으로 확정할 수 없고, 측정 불확실성을 키우는 요인이다.
- `load_for_classification`의 다른 split도 같은 이상 샘플을 재사용하므로 새로운 이상에 대한 일반화 검증은 아니다.
- MIMII 공개 데이터 결과이므로 실제 프레스 소리에서는 별도의 현장 재검증이 필요하다.
